# Mini Project: An Alert Triage Agent with LangGraph

**IT 7075C Applied AI for Cybersecurity** · Dharmin Patel

A small LangGraph agent that decides whether a security alert is **escalated** to an analyst or **closed**.
Every node is plain Python: no API key, no model, no network. The agent is deterministic, so every number
in this notebook is exact and reproduces on every run.

* **Parts 1–5**: the five TODOs (state, node, router, tool node, build and compile).
* **Parts 6–8**: run the practice graph on the course alerts, trace one alert, memory by thread id.
* **Part 9 onward**: my 12 labelled alerts, the keyword sweep, my change to the graph, the two error
  columns measured one change at a time, and the comparison with a LangChain chain.

## Part 0: Setup

In [1]:
import json, operator, csv
from collections import Counter
from typing import TypedDict, Annotated
from importlib.metadata import version

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

print("langgraph", version("langgraph"), "| langchain-core", version("langchain-core"))

langgraph 1.2.20 | langchain-core 1.7.0


### The course alerts and the lecture keyword rule

Eight practice alerts: four real incidents and four routine ones. The rule from the lecture escalates an
alert if its text contains any of five words.

In [2]:
COURSE_ALERTS = [
    {"id": "P1", "host": "WS-ENG-04", "label": "escalate",
     "text": "EDR: mimikatz sekurlsa::logonpasswords command line detected on WS-ENG-04."},
    {"id": "P2", "host": "FS-01", "label": "escalate",
     "text": "File server FS-01: 3,200 files renamed to .locked and a ransomware note README_DECRYPT.txt written."},
    {"id": "P3", "host": "DC-02", "label": "escalate",
     "text": "Windows 4720/4732: new local account 'support$' created and added to Administrators on DC-02 at 02:40."},
    {"id": "P4", "host": "FIN-WS-09", "label": "escalate",
     "text": "Proxy: 4.2 GB uploaded from FIN-WS-09 to mega.nz over 20 minutes by user k.wright."},
    {"id": "P5", "host": "WS-ENG-04", "label": "close",
     "text": "Three failed logons for t.lee on WS-ENG-04, then a successful logon at the console."},
    {"id": "P6", "host": "WS-ENG-04", "label": "close",
     "text": "Windows Update installed KB5031356 on WS-ENG-04 and scheduled a reboot."},
    {"id": "P7", "host": "FW-GUEST", "label": "close",
     "text": "Firewall: blocked a request to the URL category 'exploit kits' from the guest Wi-Fi."},
    {"id": "P8", "host": "WS-ENG-04", "label": "close",
     "text": "Defender: malware definitions updated to version 1.401.1022."},
]
KEYWORDS = ["malware", "ransomware", "mimikatz", "exploit", "brute force"]   # the lecture rule

# The tool: an asset inventory lookup. An ordinary Python function that a node calls.
ASSET_DB = {
    "WS-ENG-04": {"owner": "Engineering", "criticality": "medium"},
    "FS-01":     {"owner": "IT",          "criticality": "high"},
    "DC-02":     {"owner": "IT",          "criticality": "high"},
    "FIN-WS-09": {"owner": "Finance",     "criticality": "medium"},
}
def lookup_asset(host: str) -> dict:
    """Tool: who owns this host and how much does it matter?"""
    return ASSET_DB.get(host, {"owner": "unknown", "criticality": "unknown"})

print(len(COURSE_ALERTS), "alerts,", sum(a["label"] == "escalate" for a in COURSE_ALERTS), "real incidents")

8 alerts, 4 real incidents


## Part 1: The state (TODO 1)

The state is the one dictionary that travels through the graph. Each node reads it and returns only the
fields it changed; LangGraph merges that update into the state before the next node runs.

In [3]:
class AlertState(TypedDict, total=False):
    alert_id: str
    text: str
    host: str
    # ================= TODO 1: the fields the nodes fill in =================
    hits: list[str]     # keywords found in the text   (written by triage)
    decision: str       # "escalate" or "close"        (written by the last node)
    # =========================================================================
    asset: dict         # owner and criticality        (written by the tool node)

print(list(AlertState.__annotations__))

['alert_id', 'text', 'host', 'hits', 'decision', 'asset']


## Part 2: A node (TODO 2)

In [4]:
def triage(state: AlertState) -> dict:
    # ================= TODO 2: return only the field this node changed =================
    hits = [k for k in KEYWORDS if k in state["text"].lower()]
    return {"hits": hits}
    # ====================================================================================

print(triage({"text": "EDR: mimikatz detected"}))    # expected: {'hits': ['mimikatz']}

{'hits': ['mimikatz']}


## Part 3: The router (TODO 3)

In [5]:
def route(state: AlertState) -> str:
    # ================= TODO 3: return the label of the next branch =================
    return "escalate" if state["hits"] else "close"
    # ================================================================================

print(route({"hits": ["mimikatz"]}), route({"hits": []}))    # expected: escalate close

escalate close


## Part 4: A node that calls a tool (TODO 4)

The lookup only runs on the escalation branch: an analyst needs the owner and criticality of an escalated
host, and nobody needs them for a closed alert.

In [6]:
def escalate(state: AlertState) -> dict:
    # ================= TODO 4: call the tool, return what changed =================
    asset = lookup_asset(state["host"])
    return {"asset": asset, "decision": "escalate"}
    # ===============================================================================

def close(state: AlertState) -> dict:
    return {"decision": "close"}

print(escalate({"host": "DC-02"}))    # expected: {'asset': {'owner': 'IT', 'criticality': 'high'}, 'decision': 'escalate'}

{'asset': {'owner': 'IT', 'criticality': 'high'}, 'decision': 'escalate'}


## Part 5: Build and compile (TODO 5)

In [7]:
builder = StateGraph(AlertState)
builder.add_node("triage", triage)
builder.add_node("escalate", escalate)
builder.add_node("close", close)
# ================= TODO 5: entry point, conditional edge, compile =================
builder.add_edge(START, "triage")
builder.add_conditional_edges("triage", route, {"escalate": "escalate", "close": "close"})
# ===================================================================================
builder.add_edge("escalate", END)
builder.add_edge("close", END)
graph = builder.compile()  # TODO 5 (compile)

print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	triage(triage)
	escalate(escalate)
	close(close)
	__end__([<p>__end__</p>]):::last
	__start__ --> triage;
	triage -.-> close;
	triage -.-> escalate;
	close --> __end__;
	escalate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Part 6: Run the practice graph and score it

Three outcomes, never one accuracy figure:

* **caught**: escalated a real incident
* **missed**: closed a real incident (the attacker carries on)
* **over-escalated**: escalated routine noise (analyst hours, then alert fatigue)

In [8]:
def outcome(label: str, decision: str) -> str:
    if label == "escalate":
        return "caught" if decision == "escalate" else "MISSED"
    return "OVER-ESCALATED" if decision == "escalate" else "closed ok"

def score(alerts, decisions) -> dict:
    o = [outcome(a["label"], decisions[a["id"]]) for a in alerts]
    return {"caught": o.count("caught"), "missed": o.count("MISSED"),
            "over_escalated": o.count("OVER-ESCALATED"), "closed_ok": o.count("closed ok"),
            "missed_ids": " ".join(a["id"] for a, x in zip(alerts, o) if x == "MISSED"),
            "over_ids": " ".join(a["id"] for a, x in zip(alerts, o) if x == "OVER-ESCALATED")}

def run_v1(g, alerts) -> dict:
    decisions = {}
    for a in alerts:
        s = g.invoke({"alert_id": a["id"], "text": a["text"], "host": a["host"]})
        decisions[a["id"]] = s["decision"]
        print(f'{a["id"]:3} label={a["label"]:8} decision={s["decision"]:8} '
              f'{outcome(a["label"], s["decision"]):15} hits={s["hits"]}')
    return decisions

course_dec = run_v1(graph, COURSE_ALERTS)
print(score(COURSE_ALERTS, course_dec))

P1  label=escalate decision=escalate caught          hits=['mimikatz']
P2  label=escalate decision=escalate caught          hits=['ransomware']
P3  label=escalate decision=close    MISSED          hits=[]
P4  label=escalate decision=close    MISSED          hits=[]
P5  label=close    decision=close    closed ok       hits=[]
P6  label=close    decision=close    closed ok       hits=[]
P7  label=close    decision=escalate OVER-ESCALATED  hits=['exploit']
P8  label=close    decision=escalate OVER-ESCALATED  hits=['malware']
{'caught': 2, 'missed': 2, 'over_escalated': 2, 'closed_ok': 2, 'missed_ids': 'P3 P4', 'over_ids': 'P7 P8'}


The lecture rule catches P1 and P2, whose text names the attack, and **misses P3 and P4**, half the real
incidents. A new administrator account on a domain controller at 02:40 and a 4 GB upload to a file-sharing
site contain none of the five words. It also **over-escalates P7 and P8**, which contain "exploit" and
"malware" but describe a blocked request and a signature update. That is the starting point.

## Part 7: Trace one alert through the graph

`stream(..., stream_mode="updates")` prints what each node returned, in the order the nodes ran. This is
the state moving through the graph.

In [9]:
p3 = COURSE_ALERTS[2]
for step in graph.stream({"alert_id": p3["id"], "text": p3["text"], "host": p3["host"]}, stream_mode="updates"):
    print(step)

{'triage': {'hits': []}}
{'close': {'decision': 'close'}}


`triage` wrote `hits: []`, the router read it and returned `"close"`, so the `close` node ran and the
`escalate` node, with its lookup tool, never ran. The decision was made at the conditional edge.

## Part 8: Memory with a checkpointer and a thread id

With a checkpointer the graph saves the state after every step, filed under a `thread_id`. Calling the graph
again with the same thread id continues from the saved state. Here a thread is one *case*: all alerts about
one host. The `seen` field uses `operator.add` as a reducer, so each run appends to it instead of
overwriting it.

In [10]:
class CaseState(AlertState, total=False):
    seen: Annotated[list[str], operator.add]     # reducer: new values are appended

def triage_mem(state: CaseState) -> dict:
    return {**triage(state), "seen": [f'{state["alert_id"]}']}

mb = StateGraph(CaseState)
mb.add_node("triage", triage_mem); mb.add_node("escalate", escalate); mb.add_node("close", close)
mb.add_edge(START, "triage")
mb.add_conditional_edges("triage", route, {"escalate": "escalate", "close": "close"})
mb.add_edge("escalate", END); mb.add_edge("close", END)
mem_graph = mb.compile(checkpointer=MemorySaver())

def thread(tid): return {"configurable": {"thread_id": tid}}

for a in COURSE_ALERTS:                                   # one thread per host
    mem_graph.invoke({"alert_id": a["id"], "text": a["text"], "host": a["host"]}, thread(f'case-{a["host"]}'))

for tid in ["case-WS-ENG-04", "case-DC-02", "case-NEW-HOST"]:
    st = mem_graph.get_state(thread(tid))
    n_ckpt = len(list(mem_graph.get_state_history(thread(tid))))
    print(f'{tid:15} seen={st.values.get("seen", [])}  last decision={st.values.get("decision")}  checkpoints={n_ckpt}')

case-WS-ENG-04  seen=['P1', 'P5', 'P6', 'P8']  last decision=escalate  checkpoints=16
case-DC-02      seen=['P3']  last decision=close  checkpoints=4
case-NEW-HOST   seen=[]  last decision=None  checkpoints=0


Four of the eight alerts are about WS-ENG-04, and that thread remembers all four (`P1 P5 P6 P8`) across four
separate `invoke` calls. `case-DC-02` saw only P3, and a thread that was never used is empty. An analyst who
opens the WS-ENG-04 case sees that the same host had a mimikatz alert before the "harmless" failed logons,
which is context no single alert carries. Without a checkpointer each `invoke` starts from nothing.

---
# Part 9 onward: my alerts, my graph, my measurements

## Part 9: My 12 labelled alerts

`my_alerts.json` holds 12 alerts I wrote, based on common alert types (EDR detections, Windows Security
events, AWS CloudTrail, Suricata signatures): six I would escalate (E1–E6) and six I would close (C1–C6).
Each has a `why`. Beyond the text, each carries the structured fields a real alert has (host, user, IPs,
parent and child process, MFA), which the keyword rule ignores.

In [11]:
MY_ALERTS = json.load(open("my_alerts.json", encoding="utf-8"))

assert len(MY_ALERTS) == 12
assert sum(a["label"] == "escalate" for a in MY_ALERTS) == 6 and sum(a["label"] == "close" for a in MY_ALERTS) == 6
assert all(a["label"] in ("escalate", "close") and a["text"] and a["why"] for a in MY_ALERTS)

for a in MY_ALERTS:
    print(f'{a["id"]}  {a["label"]:8}  {a["source"]:38} {a["host"]}')

E1  escalate  EDR                                    WS-HR-03
E2  escalate  Windows Security (4625/4624)           VPN-GW-01
E3  escalate  EDR                                    FIN-APP-02
E4  escalate  AWS CloudTrail                         aws:acme-payroll
E5  escalate  EDR telemetry                          WS-SALES-14
E6  escalate  Suricata IDS                           DEV-BUILD-01
C1  close     Microsoft Defender                     WS-IT-01
C2  close     Suricata IDS                           WEB-01
C3  close     Windows Security (4625/4624)           WS-MKT-11
C4  close     PowerShell script block logging (4104) ADMIN-JUMP-01
C5  close     AWS CloudTrail                         aws:acme-prod
C6  close     EDR                                    SCCM-01


## Part 10: Keyword rules on my alerts (the original graph, one word list at a time)

The graph is the Part 5 graph, unchanged. Only the word list changes:

* **K0**: the lecture's five words
* **K1**: K0 plus five attacker-technique words (`credential, lsass, encoded, psexec, beacon`)
* **K2**: K1 plus five broad words (`powershell, admin, failed, delete, unusual`)

In [12]:
K0 = list(KEYWORDS)
K1 = K0 + ["credential", "lsass", "encoded", "psexec", "beacon"]
K2 = K1 + ["powershell", "admin", "failed", "delete", "unusual"]
WORDLISTS = {"K0": K0, "K1": K1, "K2": K2}

TOOL_CALLS = Counter()
def tool(fn):
    """Count every tool call, so the graph and the chain can be compared on work done."""
    def wrapped(*args):
        TOOL_CALLS[fn.__name__] += 1
        return fn(*args)
    wrapped.__name__ = fn.__name__
    return wrapped

def build_v1(keywords):
    """The Part 5 graph with a given word list. Same nodes, same edges."""
    def triage_k(state):
        return {"hits": [k for k in keywords if k in state["text"].lower()]}
    b = StateGraph(AlertState)
    b.add_node("triage", triage_k); b.add_node("escalate", escalate); b.add_node("close", close)
    b.add_edge(START, "triage")
    b.add_conditional_edges("triage", route, {"escalate": "escalate", "close": "close"})
    b.add_edge("escalate", END); b.add_edge("close", END)
    return b.compile()

RESULTS, DECISIONS = [], {}

def record(step, graph_name, wl, change, decisions, alerts=MY_ALERTS):
    row = {"step": step, "graph": graph_name, "keywords": wl, "change": change, **score(alerts, decisions)}
    RESULTS.append(row); DECISIONS[step] = decisions
    print(f'{step:2} {graph_name:3} {wl} | caught {row["caught"]}  missed {row["missed"]} [{row["missed_ids"]}]  '
          f'over-escalated {row["over_escalated"]} [{row["over_ids"]}]   ({change})')

for i, (name, wl) in enumerate(WORDLISTS.items(), start=1):
    g = build_v1(wl)
    dec = {a["id"]: g.invoke({"alert_id": a["id"], "text": a["text"], "host": a["host"]})["decision"]
           for a in MY_ALERTS}
    record(i, "v1", name, {"K0": "lecture word list", "K1": "+5 technique words",
                           "K2": "+5 broad words"}[name], dec)

 1 v1  K0 | caught 2  missed 4 [E3 E4 E5 E6]  over-escalated 2 [C1 C2]   (lecture word list)
 2 v1  K1 | caught 3  missed 3 [E4 E5 E6]  over-escalated 3 [C1 C2 C6]   (+5 technique words)
 3 v1  K2 | caught 3  missed 3 [E4 E5 E6]  over-escalated 5 [C1 C2 C3 C4 C6]   (+5 broad words)


* **K0** catches only E1 and E2, whose detection names say "Ransomware" and "Brute force". It misses four
  of six incidents and over-escalates both keyword hard cases (C1 EICAR "malware", C2 scanner "exploit").
* **K1** catches E3 ("credential", "lsass") at the cost of C6, the patching job that uses PsExec.
* **K2** buys **nothing** on the missed column and adds two more over-escalations (C3 "failed", C4
  "PowerShell", "ADMIN-JUMP-01"). E4, E5 and E6 stay missed at every word list.

## Part 11: My change to the graph: two new decisions and three tools

A keyword rule only sees words. What decided my labels was **context**: is there an approved change for this
host right now, is the source our own scanner, is this a privileged account coming from outside, did an
Office program start a script host, is the IP known-bad? So I changed the graph itself:

1. **Router change.** After `triage`, a keyword hit no longer goes straight to `escalate`. It goes to
   `check_suppression`, and an alert *without* a hit no longer goes straight to `close`. It goes to
   `check_context`.
2. **Two new nodes, each followed by its own conditional edge** (a second and third decision).
3. **Three new tools** (stubs I wrote): `lookup_change` (change calendar), `lookup_asset2` (asset inventory
   incl. scanners and criticality), `ip_reputation` (a small threat-intel table).

Each part can be switched on alone, so it can be measured one change at a time.

In [13]:
# ---------- tool data (stubs I wrote; in production these are CMDB, change calendar, threat intel) ----------
MY_ASSETS = {
    "WS-HR-03": "medium", "VPN-GW-01": "high", "FIN-APP-02": "high", "aws:acme-payroll": "high",
    "WS-SALES-14": "medium", "DEV-BUILD-01": "low", "WS-IT-01": "low", "WEB-01": "high",
    "WS-MKT-11": "low", "ADMIN-JUMP-01": "high", "aws:acme-prod": "high", "SCCM-01": "high",
}
APPROVED_SCANNERS = {"10.20.0.15"}
CHANGES = [
    {"id": "CHG-4471", "hosts": ["WS-IT-01"], "start": "2026-11-04T13:00:00Z", "end": "2026-11-04T17:00:00Z",
     "what": "monthly AV validation with EICAR"},
    {"id": "CHG-4502", "hosts": ["SCCM-01"], "start": "2026-11-05T22:00:00Z", "end": "2026-11-06T04:00:00Z",
     "what": "November patch deployment"},
]
REPUTATION = {"198.51.100.23": "malicious", "192.0.2.44": "malicious"}
PRIVILEGED = {"root", "s.kim", "p.shah", "svc_sccm"}
CORPORATE_PREFIXES = ("10.", "198.18.4.")
OFFICE = {"WINWORD.EXE", "EXCEL.EXE", "OUTLOOK.EXE", "POWERPNT.EXE"}
LOLBINS = {"mshta.exe", "rundll32.exe", "regsvr32.exe", "wscript.exe", "certutil.exe", "powershell.exe"}

# ---------- the tools: ordinary Python functions that nodes call ----------
@tool
def lookup_change(host: str, when: str):
    """Approved change covering this host at this time, or None."""
    for c in CHANGES:
        if host in c["hosts"] and c["start"] <= when <= c["end"]:
            return c
    return None

@tool
def lookup_asset2(name: str) -> dict:
    """Criticality of a host, and whether an IP is one of our approved scanners."""
    return {"criticality": MY_ASSETS.get(name, "unknown"), "approved_scanner": name in APPROVED_SCANNERS}

@tool
def ip_reputation(ip: str) -> str:
    return REPUTATION.get(ip, "unknown")

def is_corporate(ip): return ip.startswith(CORPORATE_PREFIXES)
print("tools ready:", [f.__name__ for f in (lookup_change, lookup_asset2, ip_reputation)])

tools ready: ['lookup_change', 'lookup_asset2', 'ip_reputation']


In [14]:
class TriageState(TypedDict, total=False):
    alert: dict
    hits: list[str]
    suppressed_by: str
    signals: list[str]
    asset: dict
    decision: str
    path: Annotated[list[str], operator.add]     # which nodes ran, appended by each node

def build_v2(keywords, suppression=True, context=True, mfa_rule=False):
    def triage2(s):
        return {"hits": [k for k in keywords if k in s["alert"]["text"].lower()], "path": ["triage"]}

    def check_suppression(s):                    # decision 2: is this keyword hit expected activity?
        a = s["alert"]
        if a.get("src_ip") and lookup_asset2(a["src_ip"])["approved_scanner"]:
            return {"suppressed_by": f'approved scanner {a["src_ip"]}', "path": ["check_suppression"]}
        c = lookup_change(a["host"], a["time"])
        return {"suppressed_by": f'{c["id"]} ({c["what"]})' if c else "", "path": ["check_suppression"]}

    def check_context(s):                        # decision 3: does an alert with no keyword have risky context?
        a, sig = s["alert"], []
        if a.get("parent", "").upper() in OFFICE and a.get("process", "").lower() in LOLBINS:
            sig.append(f'office app spawned {a["process"]}')
        for ip in (a.get("src_ip"), a.get("dst_ip")):
            if ip and ip_reputation(ip) == "malicious":
                sig.append(f"known-bad IP {ip}")
        if a.get("user") in PRIVILEGED and a.get("src_ip") and not is_corporate(a["src_ip"]):
            crit = lookup_asset2(a["host"])["criticality"]
            if crit == "high" and (not mfa_rule or a.get("mfa") is False):
                sig.append(f'privileged {a["user"]} from external IP on {crit}-criticality {a["host"]}'
                           + (" without MFA" if mfa_rule else ""))
        return {"signals": sig, "path": ["check_context"]}

    def escalate2(s):
        return {"asset": lookup_asset2(s["alert"]["host"]), "decision": "escalate", "path": ["escalate"]}
    def close2(s):
        return {"decision": "close", "path": ["close"]}

    def route_first(s):
        if s["hits"]:
            return "check_suppression" if suppression else "escalate"
        return "check_context" if context else "close"
    def route_suppression(s): return "close" if s["suppressed_by"] else "escalate"
    def route_context(s):     return "escalate" if s["signals"] else "close"

    b = StateGraph(TriageState)
    b.add_node("triage", triage2); b.add_node("escalate", escalate2); b.add_node("close", close2)
    b.add_edge(START, "triage")
    targets = {}                                 # only the branches this configuration can reach
    if not suppression: targets["escalate"] = "escalate"
    if not context:     targets["close"] = "close"
    if suppression:
        b.add_node("check_suppression", check_suppression); targets["check_suppression"] = "check_suppression"
        b.add_conditional_edges("check_suppression", route_suppression, {"escalate": "escalate", "close": "close"})
    if context:
        b.add_node("check_context", check_context); targets["check_context"] = "check_context"
        b.add_conditional_edges("check_context", route_context, {"escalate": "escalate", "close": "close"})
    b.add_conditional_edges("triage", route_first, targets)
    b.add_edge("escalate", END); b.add_edge("close", END)
    return b.compile()

print(build_v2(K1, mfa_rule=True).get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	triage(triage)
	escalate(escalate)
	close(close)
	check_suppression(check_suppression)
	check_context(check_context)
	__end__([<p>__end__</p>]):::last
	__start__ --> triage;
	check_context -.-> close;
	check_context -.-> escalate;
	check_suppression -.-> close;
	check_suppression -.-> escalate;
	triage -.-> check_context;
	triage -.-> check_suppression;
	close --> __end__;
	escalate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Part 12: Measure the change, one thing at a time

Starting from v1 with K1 (step 2), I turn on one piece at a time, then vary the word list again with the
final graph to see whether the graph change makes the word list matter less.

In [15]:
def run_v2(g, alerts=MY_ALERTS):
    return {a["id"]: g.invoke({"alert": a})["decision"] for a in alerts}

V2_STEPS = [
    (4, "K1", dict(suppression=True,  context=False),                "+ suppression node (change calendar, scanner list)"),
    (5, "K1", dict(suppression=True,  context=True),                 "+ context node (process, reputation, privileged-external)"),
    (6, "K1", dict(suppression=True,  context=True, mfa_rule=True),  "privileged-external rule requires MFA absent"),
    (7, "K0", dict(suppression=True,  context=True, mfa_rule=True),  "final graph, narrower word list K0"),
    (8, "K2", dict(suppression=True,  context=True, mfa_rule=True),  "final graph, wider word list K2"),
]
for step, wl, kw, change in V2_STEPS:
    TOOL_CALLS.clear()
    record(step, "v2", wl, change, run_v2(build_v2(WORDLISTS[wl], **kw)))
    RESULTS[-1]["tool_calls"] = sum(TOOL_CALLS.values())

 4 v2  K1 | caught 3  missed 3 [E4 E5 E6]  over-escalated 0 []   (+ suppression node (change calendar, scanner list))
 5 v2  K1 | caught 5  missed 1 [E6]  over-escalated 1 [C5]   (+ context node (process, reputation, privileged-external))
 6 v2  K1 | caught 5  missed 1 [E6]  over-escalated 0 []   (privileged-external rule requires MFA absent)
 7 v2  K0 | caught 4  missed 2 [E3 E6]  over-escalated 0 []   (final graph, narrower word list K0)
 8 v2  K2 | caught 5  missed 1 [E6]  over-escalated 2 [C3 C4]   (final graph, wider word list K2)


* **Step 4** (suppression only): over-escalations fall from 3 to **0**. C1 is covered by CHG-4471, C6 by
  CHG-4502, and C2 comes from the approved scanner. The missed column does not move, because suppression only
  ever closes alerts.
* **Step 5** (+ context): E4 (root from outside on payroll) and E5 (Outlook → mshta → known-bad IP) are now
  caught, so missed falls from 3 to **1**. It costs **one** new over-escalation: C5, the travelling engineer,
  who is also privileged, external, and on a critical account.
* **Step 6** (the privileged-external rule also needs MFA to be absent): C5 closes again, E4 stays caught.
* **Steps 7 and 8**: with the final graph, K0 loses E3 again, and K2's extra words still bring C3 and C4 back,
  because neither has a change ticket. Widening the word list still only adds noise.
* **E6 (DNS tunnelling) is missed at every one of the eight settings.**

## Part 13: The results table → `results_mine.csv`

In [16]:
cols = ["step", "graph", "keywords", "change", "caught", "missed", "over_escalated", "closed_ok",
        "missed_ids", "over_ids", "tool_calls"]
with open("results_mine.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
    for r in RESULTS:
        w.writerow({c: r.get(c, "") for c in cols})

import pandas as pd
pd.set_option("display.max_colwidth", 70)
pd.read_csv("results_mine.csv", dtype=str).fillna("")

,step,graph,keywords,change,caught,missed,over_escalated,closed_ok,missed_ids,over_ids,tool_calls
0,1,v1,K0,lecture word list,2,4,2,4,E3 E4 E5 E6,C1 C2,
1,2,v1,K1,+5 technique words,3,3,3,3,E4 E5 E6,C1 C2 C6,
2,3,v1,K2,+5 broad words,3,3,5,1,E4 E5 E6,C1 C2 C3 C4 C6,
3,4,v2,K1,"+ suppression node (change calendar, scanner list)",3,3,0,6,E4 E5 E6,,11
4,5,v2,K1,"+ context node (process, reputation, privileged-external)",5,1,1,5,E6,C5,21
5,6,v2,K1,privileged-external rule requires MFA absent,5,1,0,6,E6,,20
6,7,v2,K0,"final graph, narrower word list K0",4,2,0,6,E3 E6,,17
7,8,v2,K2,"final graph, wider word list K2",5,1,2,4,E6,C3 C4,24


## Part 14: Which alerts are wrong at which setting

One row per alert, one column per step. A ✗ is a wrong decision (missed or over-escalated).

In [17]:
labels = {a["id"]: a["label"] for a in MY_ALERTS}
matrix = pd.DataFrame({f'{r["step"]}:{r["graph"]}/{r["keywords"]}':
                       {i: ("✗" if (d == "escalate") != (labels[i] == "escalate") else "·")
                        for i, d in DECISIONS[r["step"]].items()} for r in RESULTS})
matrix.insert(0, "label", pd.Series(labels))
display(matrix)

v1_cols = [c for c in matrix.columns if ":v1/" in c]
print("wrong at every keyword setting (v1):", [i for i in matrix.index if all(matrix.loc[i, c] == "✗" for c in v1_cols)])
print("wrong at every setting (v1 and v2):", [i for i in matrix.index if all(matrix.loc[i, c] == "✗" for c in matrix.columns[1:])])

,label,1:v1/K0,2:v1/K1,3:v1/K2,4:v2/K1,5:v2/K1,6:v2/K1,7:v2/K0,8:v2/K2
E1,escalate,·,·,·,·,·,·,·,·
E2,escalate,·,·,·,·,·,·,·,·
E3,escalate,✗,·,·,·,·,·,✗,·
E4,escalate,✗,✗,✗,✗,·,·,·,·
E5,escalate,✗,✗,✗,✗,·,·,·,·
E6,escalate,✗,✗,✗,✗,✗,✗,✗,✗
C1,close,✗,✗,✗,·,·,·,·,·
C2,close,✗,✗,✗,·,·,·,·,·
C3,close,·,·,✗,·,·,·,·,✗
C4,close,·,·,✗,·,·,·,·,✗


wrong at every keyword setting (v1): ['E4', 'E5', 'E6', 'C1', 'C2']
wrong at every setting (v1 and v2): ['E6']


## Part 15: Trace a caught alert and a missed one through the final graph

The `path` field (with an `operator.add` reducer) records which nodes ran, and `stream` shows what each one
returned.

In [18]:
FINAL = build_v2(K1, suppression=True, context=True, mfa_rule=True)
by_id = {a["id"]: a for a in MY_ALERTS}

for aid in ["E4", "C1", "E6"]:
    print(f"=== {aid} (label: {by_id[aid]['label']}) ===")
    for step in FINAL.stream({"alert": by_id[aid]}, stream_mode="updates"):
        print("  ", step)
    print()

=== E4 (label: escalate) ===
   {'triage': {'hits': [], 'path': ['triage']}}
   {'check_context': {'signals': ['privileged root from external IP on high-criticality aws:acme-payroll without MFA'], 'path': ['check_context']}}
   {'escalate': {'asset': {'criticality': 'high', 'approved_scanner': False}, 'decision': 'escalate', 'path': ['escalate']}}

=== C1 (label: close) ===
   {'triage': {'hits': ['malware'], 'path': ['triage']}}
   {'check_suppression': {'suppressed_by': 'CHG-4471 (monthly AV validation with EICAR)', 'path': ['check_suppression']}}
   {'close': {'decision': 'close', 'path': ['close']}}

=== E6 (label: escalate) ===
   {'triage': {'hits': [], 'path': ['triage']}}
   {'check_context': {'signals': [], 'path': ['check_context']}}
   {'close': {'decision': 'close', 'path': ['close']}}



* **E4, caught.** `triage` finds no keyword, so the first router sends it to `check_context`. That node
  calls the asset tool (payroll is high criticality) and finds a privileged account (`root`) from an external
  IP without MFA. `route_context` sends it to `escalate`. The v1 graph closed this alert at the first edge.
* **C1, correctly closed.** `triage` finds "malware", so it goes to `check_suppression`, where
  `lookup_change` finds CHG-4471 covering WS-IT-01 at 14:20, and `route_suppression` sends it to `close`.
* **E6, missed.** No keyword, no Office parent, no known-bad IP (the domain is new), no privileged user. Every
  check the graph knows how to make comes back empty, so it closes.

## Part 16: The graph compared with a LangChain chain

The same three checks written as a LangChain chain (`RunnableLambda` piped with `|`). A chain runs every
stage on every input, so each check has to run and then be ignored when it does not apply. I count tool calls
for both on the same 12 alerts.

In [19]:
from langchain_core.runnables import RunnableLambda

def c_triage(a):   return {"alert": a, "hits": [k for k in K1 if k in a["text"].lower()]}
def c_suppress(s):
    a = s["alert"]
    scanner = bool(a.get("src_ip")) and lookup_asset2(a["src_ip"])["approved_scanner"]
    return {**s, "suppressed": scanner or lookup_change(a["host"], a["time"]) is not None}
def c_context(s):
    a, sig = s["alert"], []
    if a.get("parent", "").upper() in OFFICE and a.get("process", "").lower() in LOLBINS: sig.append("office-lolbin")
    for ip in (a.get("src_ip"), a.get("dst_ip")):
        if ip and ip_reputation(ip) == "malicious": sig.append("bad-ip")
    crit = lookup_asset2(a["host"])["criticality"]
    if a.get("user") in PRIVILEGED and a.get("src_ip") and not is_corporate(a["src_ip"]) \
            and crit == "high" and a.get("mfa") is False:
        sig.append("priv-external")
    return {**s, "signals": sig}
def c_decide(s):
    esc = (s["hits"] and not s["suppressed"]) or (not s["hits"] and s["signals"])
    return "escalate" if esc else "close"

chain = RunnableLambda(c_triage) | RunnableLambda(c_suppress) | RunnableLambda(c_context) | RunnableLambda(c_decide)

TOOL_CALLS.clear(); chain_dec = {a["id"]: chain.invoke(a) for a in MY_ALERTS}; chain_calls = dict(TOOL_CALLS)
TOOL_CALLS.clear(); graph_dec = run_v2(FINAL);                                 graph_calls = dict(TOOL_CALLS)

print("same decisions on all 12:", chain_dec == graph_dec)
print("chain tool calls:", chain_calls, "total", sum(chain_calls.values()))
print("graph tool calls:", graph_calls, "total", sum(graph_calls.values()))
print("\nnodes the graph ran per alert:")
for a in MY_ALERTS:
    print(f'  {a["id"]}: {" -> ".join(FINAL.invoke({"alert": a})["path"])}')

same decisions on all 12: True
chain tool calls: {'lookup_change': 11, 'lookup_asset2': 19, 'ip_reputation': 8} total 38
graph tool calls: {'lookup_change': 5, 'lookup_asset2': 10, 'ip_reputation': 5} total 20

nodes the graph ran per alert:
  E1: triage -> check_suppression -> escalate
  E2: triage -> check_suppression -> escalate
  E3: triage -> check_suppression -> escalate
  E4: triage -> check_context -> escalate
  E5: triage -> check_context -> escalate
  E6: triage -> check_context -> close
  C1: triage -> check_suppression -> close
  C2: triage -> check_suppression -> close
  C3: triage -> check_context -> close
  C4: triage -> check_context -> close
  C5: triage -> check_context -> close
  C6: triage -> check_suppression -> close


The decisions are identical, so the chain is not *wrong*; the difference is in what runs. The chain made
**38** tool calls and the graph **20** for the same 12 decisions. The chain does every check for every alert and folds the branching into one `if` at the end, where it is invisible. The
graph's routing is in its edges: a keyword hit never runs the context check, and each alert's `path` shows
exactly which branch it took. The decision logic is what this project studies, and in the graph it is
visible, drawable and traceable. And a chain has no checkpointer: the Part 8 case memory, keyed by thread
id, is something a compiled graph provides and a chain does not.

## Part 17: Does my graph change transfer? (the course alerts)

The rules in Part 11 were written after I had seen my 12 alerts. A fair check is to run the final graph on
alerts it was not designed for: the eight course alerts. The course alerts have no IP, user or process
fields, and their hosts are not in my inventory or change calendar.

In [20]:
course_v2 = {a["id"]: FINAL.invoke({"alert": {**a, "time": "2026-11-01T00:00:00Z"}})["decision"] for a in COURSE_ALERTS}
print("v1 K0 on course alerts:", {k: v for k, v in score(COURSE_ALERTS, course_dec).items()})
print("v2 K1 on course alerts:", {k: v for k, v in score(COURSE_ALERTS, course_v2).items()})

v1 K0 on course alerts: {'caught': 2, 'missed': 2, 'over_escalated': 2, 'closed_ok': 2, 'missed_ids': 'P3 P4', 'over_ids': 'P7 P8'}
v2 K1 on course alerts: {'caught': 2, 'missed': 2, 'over_escalated': 2, 'closed_ok': 2, 'missed_ids': 'P3 P4', 'over_ids': 'P7 P8'}


No improvement: the context checks need structured fields the course alerts do not have, and the
suppression checks need tickets that do not exist for those hosts. The change is not a general fix. It
works where the enrichment data exists, which is the real lesson: in a SOC the value is in the change
calendar, the asset inventory and the threat intel behind the tools, not in the graph shape alone.

## Summary

| setting | caught | missed | over-escalated |
|---|---|---|---|
| v1, lecture words (K0) | 2 | 4 | 2 |
| v1, K1 | 3 | 3 | 3 |
| v1, K2 | 3 | 3 | 5 |
| v2, K1 + suppression + context (MFA rule), **deployed** | 5 | 1 (E6) | 0 |

Twelve alerts means each alert is a sixth of a column, and the rules were written after reading the alerts.
These numbers show *which* alert each change fixes and what it costs; they are not an estimate of how the
agent would perform on a real queue.